<a href="https://colab.research.google.com/github/arielabade/carbon/blob/main/baselineEvaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:



import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.utils import Sequence
import gc

# Parâmetros
chunk_size = 1000
max_len_limit = 500
batch_size = 16
epochs = 20
random_state = 42

# Caminhos dos arquivos, incluindo o NEB no treinamento
file_paths_train = [
    "/content/ANKRD1_test_CORRECTED.csv",
    "/content/B2MFIX_test_CORRECTED.csv",
    "/content/PPIAFIX2_test_CORRECTED.csv",
    "/content/GADPH_test_CORRECTED.csv",
    "/content/PGK1_test_CORRECTED.csv",
    "/content/RPLA13A_test_CORRECTED.csv",
    "/content/TTN_test_CORRECTED.csv",
    "/content/NEB_test_test_CORRECTED.csv"
]

# Função para processar chunks
def process_chunk(chunk):
    sequences = chunk['sequence'].values
    labels = chunk['exon_intron_flag'].values
    return sequences, labels

# Inicializando o tokenizer
tokenizer = Tokenizer(char_level=True)
sequences_list = []
labels_list = []

# Processando os arquivos CSV em chunks para treino
for file_path in file_paths_train:
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        sequences, labels = process_chunk(chunk)
        tokenizer.fit_on_texts(sequences)
        sequences_list.extend(sequences)
        labels_list.extend(labels)
        gc.collect()

# Tokenização e padding das sequências
encoded_sequences = tokenizer.texts_to_sequences(sequences_list)
max_len = min(max(len(seq) for seq in encoded_sequences), max_len_limit)
padded_sequences = pad_sequences(encoded_sequences, maxlen=max_len, padding='post')
labels = np.array(labels_list, dtype=np.int8)

# Separando os dados em treino (80%), validação (10%) e teste (10%)
X_train, X_temp, y_train, y_temp = train_test_split(padded_sequences, labels, test_size=0.2, random_state=random_state)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=random_state)

# Construção do modelo RNN com GRU
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 32

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    GRU(32, return_sequences=True),  # Primeira camada GRU
    Dropout(0.2),
    GRU(32),  # Segunda camada GRU
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Classe Data Generator
class DataGenerator(Sequence):
    def __init__(self, sequences, labels, batch_size):
        self.sequences = sequences
        self.labels = labels
        self.batch_size = batch_size
        self.indexes = np.arange(len(self.sequences))

    def __len__(self):
        return int(np.ceil(len(self.sequences) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        return self.sequences[batch_indexes], self.labels[batch_indexes]

# Criando os geradores de dados
train_generator = DataGenerator(X_train, y_train, batch_size)
validation_generator = DataGenerator(X_val, y_val, batch_size)

# Treinamento do modelo
model.fit(train_generator, epochs=epochs, validation_data=validation_generator)

# Função para calcular métricas de avaliação
def evaluate_model(generator, y_true):
    predictions = model.predict(generator)
    predictions = (predictions > 0.5).astype(int)

    # Métricas de classificação
    accuracy = accuracy_score(y_true, predictions)
    precision = precision_score(y_true, predictions)
    sensitivity = recall_score(y_true, predictions)

    # Cálculo da especificidade
    tn = np.sum((y_true == 0) & (predictions == 0))
    fp = np.sum((y_true == 0) & (predictions == 1))
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    # F1-score
    f1 = f1_score(y_true, predictions)

    return accuracy, precision, sensitivity, specificity, f1

# Avaliação no conjunto de teste
test_generator = DataGenerator(X_test, y_test, batch_size)
accuracy, precision, sensitivity, specificity, f1 = evaluate_model(test_generator, y_test)

# Exibindo métricas
print(f'Test Accuracy: {accuracy * 100:.2f}%')
print(f'Test Precision: {precision:.2f}')
print(f'Test Sensitivity (Recall): {sensitivity:.2f}')
print(f'Test Specificity: {specificity:.2f}')
print(f'Test F1 Score: {f1:.2f}')


Epoch 1/20


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


499/499 ━━━━━━━━━━━━━━━━━━━━ 264s 516ms/step - accuracy: 0.6479 - loss: 0.6233 - val_accuracy: 0.6797 - val_loss: 0.5655
Epoch 2/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 244s 490ms/step - accuracy: 0.6950 - loss: 0.5447 - val_accuracy: 0.6325 - val_loss: 0.5526
Epoch 3/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 255s 476ms/step - accuracy: 0.6485 - loss: 0.6250 - val_accuracy: 0.6797 - val_loss: 0.5643
Epoch 4/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 266s 483ms/step - accuracy: 0.6705 - loss: 0.5684 - val_accuracy: 0.6817 - val_loss: 0.5550
Epoch 5/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 260s 480ms/step - accuracy: 0.6701 - loss: 0.5659 - val_accuracy: 0.6797 - val_loss: 0.5565
Epoch 6/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 263s 481ms/step - accuracy: 0.6851 - loss: 0.5492 - val_accuracy: 0.6837 - val_loss: 0.5516
Epoch 7/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 238s 477ms/step - accuracy: 0.6735 - loss: 0.5532 - val_accuracy: 0.6837 - val_loss: 0.5498
Epoch 8/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 262s 478ms/step - accuracy: 0.6920 - loss: 0.53

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.utils import Sequence
import gc

# Parâmetros
chunk_size = 1000
max_len_limit = 500
batch_size = 16
epochs = 20
random_state = 42

# Caminhos dos arquivos, incluindo o NEB no treinamento
file_paths_train = [
    "/content/ANKRD1_test_CORRECTED.csv",
    "/content/B2MFIX_test_CORRECTED.csv",
    "/content/PPIAFIX2_test_CORRECTED.csv",
    "/content/GADPH_test_CORRECTED.csv",
    "/content/PGK1_test_CORRECTED.csv",
    "/content/RPLA13A_test_CORRECTED.csv",
    "/content/TTN_test_CORRECTED.csv",
    "/content/NEB_test_test_CORRECTED.csv"  # Incluindo NEB no treino
]

# Função para processar chunks
def process_chunk(chunk):
    sequences = chunk['sequence'].values
    labels = chunk['exon_intron_flag'].values
    return sequences, labels

# Inicializando o tokenizer
tokenizer = Tokenizer(char_level=True)
sequences_list = []  # Inicialização correta das variáveis
labels_list = []

# Processando os arquivos CSV em chunks para treino (incluindo NEB)
for file_path in file_paths_train:
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        sequences, labels = process_chunk(chunk)
        tokenizer.fit_on_texts(sequences)
        sequences_list.extend(sequences)
        labels_list.extend(labels)
        gc.collect()

# Tokenização e padding das sequências
encoded_sequences = tokenizer.texts_to_sequences(sequences_list)
max_len = min(max(len(seq) for seq in encoded_sequences), max_len_limit)
padded_sequences = pad_sequences(encoded_sequences, maxlen=max_len, padding='post')
labels = np.array(labels_list, dtype=np.int8)

# Separando os dados em treino (80%), validação (10%) e teste (10%)
X_train, X_temp, y_train, y_temp = train_test_split(padded_sequences, labels, test_size=0.2, random_state=random_state)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=random_state)

# Construção do modelo Bidirectional LSTM
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 32

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    Bidirectional(LSTM(32, return_sequences=True)),
    Dropout(0.2),
    Bidirectional(LSTM(32)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Classe Data Generator
class DataGenerator(Sequence):
    def __init__(self, sequences, labels, batch_size):
        self.sequences = sequences
        self.labels = labels
        self.batch_size = batch_size
        self.indexes = np.arange(len(self.sequences))

    def __len__(self):
        return int(np.ceil(len(self.sequences) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        return self.sequences[batch_indexes], self.labels[batch_indexes]

# Criando os geradores de dados
train_generator = DataGenerator(X_train, y_train, batch_size)
validation_generator = DataGenerator(X_val, y_val, batch_size)

# Treinamento do modelo
model.fit(train_generator, epochs=epochs, validation_data=validation_generator)

# Função para calcular métricas de avaliação
def evaluate_model(generator, y_true):
    predictions = model.predict(generator)
    predictions = (predictions > 0.5).astype(int)
    accuracy = accuracy_score(y_true, predictions)
    precision = precision_score(y_true, predictions)
    sensitivity = recall_score(y_true, predictions)
    f1 = f1_score(y_true, predictions)
    return accuracy, precision, sensitivity, f1

# Avaliação nos arquivos de treino/validação
results = {
    "File": [], "Validation ACC": [], "Validation PRE": [], "Validation SEN": [], "Validation F1S": [],
    "Test ACC": [], "Test PRE": [], "Test SEN": [], "Test F1S": []
}

for test_file in file_paths_train:
    test_data = pd.read_csv(test_file)
    test_sequences = test_data['sequence'].values
    test_labels = test_data['exon_intron_flag'].values

    # Tokenização e padding das sequências de teste
    encoded_test_sequences = tokenizer.texts_to_sequences(test_sequences)
    padded_test_sequences = pad_sequences(encoded_test_sequences, maxlen=max_len, padding='post')

    # Avaliação nos dados de validação
    val_accuracy, val_precision, val_sensitivity, val_f1 = evaluate_model(validation_generator, y_val)

    # Avaliação nos dados de teste
    test_predictions = model.predict(padded_test_sequences)
    test_predictions = (test_predictions > 0.5).astype(int)
    test_accuracy = accuracy_score(test_labels, test_predictions)
    test_precision = precision_score(test_labels, test_predictions)
    test_sensitivity = recall_score(test_labels, test_predictions)
    test_f1_score = f1_score(test_labels, test_predictions)

    # Armazenando os resultados
    results["File"].append(test_file)
    results["Validation ACC"].append(val_accuracy * 100)
    results["Validation PRE"].append(val_precision)
    results["Validation SEN"].append(val_sensitivity)
    results["Validation F1S"].append(val_f1)
    results["Test ACC"].append(test_accuracy * 100)
    results["Test PRE"].append(test_precision)
    results["Test SEN"].append(test_sensitivity)
    results["Test F1S"].append(test_f1_score)

    print(f'Results for {test_file}:')
    print(f'Validation Accuracy: {val_accuracy * 100:.2f}%')
    print(f'Test Accuracy: {test_accuracy * 100:.2f}%')
    print(f'Test Precision: {test_precision:.2f}')
    print(f'Test Sensitivity: {test_sensitivity:.2f}')
    print(f'Test F1 Score: {test_f1_score:.2f}')
    print('---')

# Exibindo todos os resultados
results_df = pd.DataFrame(results)
print(results_df)

# Calculando as médias totais
numeric_columns = results_df.select_dtypes(include=[np.number])
total_avg = numeric_columns.mean(axis=0)

# Adicionando as métricas totais ao DataFrame
total_metrics = pd.DataFrame({
    "File": ["Total"],
    "Validation ACC": [total_avg["Validation ACC"]],
    "Validation PRE": [total_avg["Validation PRE"]],
    "Validation SEN": [total_avg["Validation SEN"]],
    "Validation F1S": [total_avg["Validation F1S"]],
    "Test ACC": [total_avg["Test ACC"]],
    "Test PRE": [total_avg["Test PRE"]],
    "Test SEN": [total_avg["Test SEN"]],
    "Test F1S": [total_avg["Test F1S"]]
})

# Concatenando as métricas totais ao DataFrame de resultados
results_df = pd.concat([results_df, total_metrics], ignore_index=True)
print("Results with Total Metrics:")
print(results_df)

# Avaliação combinada
combined_test_labels = np.concatenate([test_labels for test_file in file_paths_train])
combined_test_sequences = np.concatenate([padded_test_sequences for test_file in file_paths_train])
test_predictions = model.predict(combined_test_sequences)
test_predictions = (test_predictions > 0.5).astype(int)

# Calculando métricas combinadas
test_accuracy = accuracy_score(combined_test_labels, test_predictions)
test_precision = precision_score(combined_test_labels, test_predictions)
test_sensitivity = recall_score(combined_test_labels, test_predictions)
test_f1_score = f1_score(combined_test_labels, test_predictions)

print(f'Test Accuracy (combined files): {test_accuracy * 100:.2f}%')
print(f'Test Precision (combined files): {test_precision:.2f}')
print(f'Test Sensitivity (combined files): {test_sensitivity:.2f}')
print(f'Test F1 Score (combined files): {test_f1_score:.2f}')

# Salvando o modelo treinado
model.save("/content/lstm_modelBILSTMTEST.h5")
print("Modelo salvo como lstm_model.h5")

# Montando o Google Drive e copiando o modelo para lá
from google.colab import drive
drive.mount('/content/drive')
!cp /content/lstm_model.h5 /content/drive/MyDrive/lstm_model.h5
print("Modelo salvo no Google Drive como lstm_model.h5")



Epoch 1/20


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


499/499 ━━━━━━━━━━━━━━━━━━━━ 364s 712ms/step - accuracy: 0.8162 - loss: 0.3980 - val_accuracy: 0.9729 - val_loss: 0.0900
Epoch 2/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 376s 700ms/step - accuracy: 0.9568 - loss: 0.1287 - val_accuracy: 0.9408 - val_loss: 0.1540
Epoch 3/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 354s 710ms/step - accuracy: 0.9251 - loss: 0.1714 - val_accuracy: 0.9779 - val_loss: 0.0660
Epoch 4/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 356s 713ms/step - accuracy: 0.9707 - loss: 0.0814 - val_accuracy: 0.9829 - val_loss: 0.0433
Epoch 5/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 354s 709ms/step - accuracy: 0.9676 - loss: 0.0892 - val_accuracy: 0.9819 - val_loss: 0.0719
Epoch 6/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 347s 696ms/step - accuracy: 0.9681 - loss: 0.0809 - val_accuracy: 0.9900 - val_loss: 0.0294
Epoch 7/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 352s 705ms/step - accuracy: 0.9890 - loss: 0.0355 - val_accuracy: 0.9498 - val_loss: 0.1285
Epoch 8/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 383s 707ms/step - accuracy: 0.9636 - loss: 0.09

Test Accuracy (combined files): 99.82%
Test Precision (combined files): 1.00
Test Sensitivity (combined files): 1.00
Test F1 Score (combined files): 1.00
Modelo salvo como lstm_model.h5


ValueError: mount failed

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.utils import Sequence
import gc

# Parâmetros
chunk_size = 1000
max_len_limit = 500
batch_size = 16
epochs = 20
random_state = 42

# Caminhos dos arquivos, incluindo o NEB no treinamento
file_paths_train = [
    "/content/ANKRD1_test_CORRECTED.csv",
    "/content/B2MFIX_test_CORRECTED.csv",
    "/content/PPIAFIX2_test_CORRECTED.csv",
    "/content/GADPH_test_CORRECTED.csv",
    "/content/PGK1_test_CORRECTED.csv",
    "/content/RPLA13A_test_CORRECTED.csv",
    "/content/TTN_test_CORRECTED.csv",
    "/content/NEB_test_test_CORRECTED.csv"  # Incluindo NEB no treino
]

# Função para processar chunks
def process_chunk(chunk):
    sequences = chunk['sequence'].values
    labels = chunk['exon_intron_flag'].values
    return sequences, labels

# Inicializando o tokenizer
tokenizer = Tokenizer(char_level=True)
sequences_list = []  # Inicialização correta das variáveis
labels_list = []

# Processando os arquivos CSV em chunks para treino (incluindo NEB)
for file_path in file_paths_train:
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        sequences, labels = process_chunk(chunk)
        tokenizer.fit_on_texts(sequences)
        sequences_list.extend(sequences)
        labels_list.extend(labels)
        gc.collect()

# Tokenização e padding das sequências
encoded_sequences = tokenizer.texts_to_sequences(sequences_list)
max_len = min(max(len(seq) for seq in encoded_sequences), max_len_limit)
padded_sequences = pad_sequences(encoded_sequences, maxlen=max_len, padding='post')
labels = np.array(labels_list, dtype=np.int8)

# Separando os dados em treino (80%), validação (10%) e teste (10%)
X_train, X_temp, y_train, y_temp = train_test_split(padded_sequences, labels, test_size=0.2, random_state=random_state)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=random_state)

# Construção do modelo Bidirectional LSTM
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 32

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    Bidirectional(LSTM(32, return_sequences=True)),
    Dropout(0.2),
    Bidirectional(LSTM(32)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Classe Data Generator
class DataGenerator(Sequence):
    def __init__(self, sequences, labels, batch_size):
        self.sequences = sequences
        self.labels = labels
        self.batch_size = batch_size
        self.indexes = np.arange(len(self.sequences))

    def __len__(self):
        return int(np.ceil(len(self.sequences) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        return self.sequences[batch_indexes], self.labels[batch_indexes]

# Criando os geradores de dados
train_generator = DataGenerator(X_train, y_train, batch_size)
validation_generator = DataGenerator(X_val, y_val, batch_size)

# Treinamento do modelo
model.fit(train_generator, epochs=epochs, validation_data=validation_generator)

# Função para calcular métricas de avaliação, incluindo a especificidade
def evaluate_model(generator, y_true):
    predictions = model.predict(generator)
    predictions = (predictions > 0.5).astype(int)
    accuracy = accuracy_score(y_true, predictions)
    precision = precision_score(y_true, predictions)
    sensitivity = recall_score(y_true, predictions)
    f1 = f1_score(y_true, predictions)

    # Cálculo da especificidade
    tn, fp, fn, tp = confusion_matrix(y_true, predictions).ravel()
    specificity = tn / (tn + fp)

    return accuracy, precision, sensitivity, f1, specificity

# Avaliação nos arquivos de treino/validação
results = {
    "File": [], "Validation ACC": [], "Validation PRE": [], "Validation SEN": [], "Validation F1S": [], "Validation SPEC": [],
    "Test ACC": [], "Test PRE": [], "Test SEN": [], "Test F1S": [], "Test SPEC": []
}

for test_file in file_paths_train:
    test_data = pd.read_csv(test_file)
    test_sequences = test_data['sequence'].values
    test_labels = test_data['exon_intron_flag'].values

    # Tokenização e padding das sequências de teste
    encoded_test_sequences = tokenizer.texts_to_sequences(test_sequences)
    padded_test_sequences = pad_sequences(encoded_test_sequences, maxlen=max_len, padding='post')

    # Avaliação nos dados de validação
    val_accuracy, val_precision, val_sensitivity, val_f1, val_specificity = evaluate_model(validation_generator, y_val)

    # Avaliação nos dados de teste
    test_predictions = model.predict(padded_test_sequences)
    test_predictions = (test_predictions > 0.5).astype(int)
    test_accuracy = accuracy_score(test_labels, test_predictions)
    test_precision = precision_score(test_labels, test_predictions)
    test_sensitivity = recall_score(test_labels, test_predictions)
    test_f1_score = f1_score(test_labels, test_predictions)

    # Especificidade para o conjunto de teste
    tn, fp, fn, tp = confusion_matrix(test_labels, test_predictions).ravel()
    test_specificity = tn / (tn + fp)

    # Armazenando os resultados
    results["File"].append(test_file)
    results["Validation ACC"].append(val_accuracy * 100)
    results["Validation PRE"].append(val_precision)
    results["Validation SEN"].append(val_sensitivity)
    results["Validation F1S"].append(val_f1)
    results["Validation SPEC"].append(val_specificity)
    results["Test ACC"].append(test_accuracy * 100)
    results["Test PRE"].append(test_precision)
    results["Test SEN"].append(test_sensitivity)
    results["Test F1S"].append(test_f1_score)
    results["Test SPEC"].append(test_specificity)

    print(f'Results for {test_file}:')
    print(f'Validation Accuracy: {val_accuracy * 100:.2f}%')
    print(f'Test Accuracy: {test_accuracy * 100:.2f}%')
    print(f'Test Precision: {test_precision:.2f}')
    print(f'Test Sensitivity: {test_sensitivity:.2f}')
    print(f'Test F1 Score: {test_f1_score:.2f}')
    print(f'Test Specificity: {test_specificity:.2f}')
    print('---')

# Exibindo todos os resultados
results_df = pd.DataFrame(results)
print(results_df)

# Calculando as médias totais
numeric_columns = results_df.select_dtypes(include=[np.number])
total_avg = numeric_columns.mean(axis=0)

# Adicionando as métricas totais ao DataFrame
total_metrics = pd.DataFrame({
    "File": ["Total"],
    "Validation ACC": [total_avg["Validation ACC"]],
    "Validation PRE": [total_avg["Validation PRE"]],
    "Validation SEN": [total_avg["Validation SEN"]],
    "Validation F1S": [total_avg["Validation F1S"]],
    "Validation SPEC": [total_avg["Validation SPEC"]],
    "Test ACC": [total_avg["Test ACC"]],
    "Test PRE": [total_avg["Test PRE"]],
    "Test SEN": [total_avg["Test SEN"]],
    "Test F1S": [total_avg["Test F1S"]],
    "Test SPEC": [total_avg["Test SPEC"]]
})

# Concatenando as métricas totais ao DataFrame de resultados
results_df = pd.concat([results_df, total_metrics], ignore_index=True)
print("Results with Total Metrics:")
print(results_df)


Epoch 1/20


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


499/499 ━━━━━━━━━━━━━━━━━━━━ 362s 711ms/step - accuracy: 0.7920 - loss: 0.4474 - val_accuracy: 0.9598 - val_loss: 0.1320
Epoch 2/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 382s 712ms/step - accuracy: 0.9593 - loss: 0.1143 - val_accuracy: 0.9839 - val_loss: 0.0664
Epoch 3/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 384s 716ms/step - accuracy: 0.9510 - loss: 0.1529 - val_accuracy: 0.9649 - val_loss: 0.1242
Epoch 4/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 353s 707ms/step - accuracy: 0.9438 - loss: 0.1349 - val_accuracy: 0.9819 - val_loss: 0.0514
Epoch 5/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 383s 708ms/step - accuracy: 0.9648 - loss: 0.1060 - val_accuracy: 0.9819 - val_loss: 0.0792
Epoch 6/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 381s 706ms/step - accuracy: 0.9580 - loss: 0.1205 - val_accuracy: 0.9729 - val_loss: 0.0756
Epoch 7/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 352s 705ms/step - accuracy: 0.9706 - loss: 0.0826 - val_accuracy: 0.9880 - val_loss: 0.0372
Epoch 8/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 352s 707ms/step - accuracy: 0.9811 - loss: 0.05

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.utils import Sequence
import gc

# Parâmetros
chunk_size = 1000
max_len_limit = 500
batch_size = 16
epochs = 2
random_state = 42

# Caminhos dos arquivos, incluindo o NEB no treinamento
file_paths_train = [
    "/content/ANKRD1_test_CORRECTED.csv",
    "/content/B2MFIX_test_CORRECTED.csv",
    "/content/PPIAFIX2_test_CORRECTED.csv",
    "/content/GADPH_test_CORRECTED.csv",
    "/content/PGK1_test_CORRECTED.csv",
    "/content/RPLA13A_test_CORRECTED.csv",
    "/content/TTN_test_CORRECTED.csv",
    "/content/NEB_test_test_CORRECTED.csv"
]

# Função para processar chunks
def process_chunk(chunk):
    sequences = chunk['sequence'].values
    labels = chunk['exon_intron_flag'].values
    return sequences, labels

# Inicializando o tokenizer
tokenizer = Tokenizer(char_level=True)
sequences_list = []
labels_list = []

# Processando os arquivos CSV em chunks para treino
for file_path in file_paths_train:
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        sequences, labels = process_chunk(chunk)
        tokenizer.fit_on_texts(sequences)
        sequences_list.extend(sequences)
        labels_list.extend(labels)
        gc.collect()

# Tokenização e padding das sequências
encoded_sequences = tokenizer.texts_to_sequences(sequences_list)
max_len = min(max(len(seq) for seq in encoded_sequences), max_len_limit)
padded_sequences = pad_sequences(encoded_sequences, maxlen=max_len, padding='post')
labels = np.array(labels_list, dtype=np.int8)

# Separando os dados em treino (80%), validação (10%) e teste (10%)
X_train, X_temp, y_train, y_temp = train_test_split(padded_sequences, labels, test_size=0.2, random_state=random_state)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=random_state)

from tensorflow.keras.layers import LSTM, Bidirectional

# Construção do modelo RNN com LSTM bidirecional
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 32

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    Bidirectional(LSTM(32, return_sequences=True)),
    Dropout(0.2),
    Bidirectional(LSTM(32)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Classe Data Generator
class DataGenerator(Sequence):
    def __init__(self, sequences, labels, batch_size):
        self.sequences = sequences
        self.labels = labels
        self.batch_size = batch_size
        self.indexes = np.arange(len(self.sequences))

    def __len__(self):
        return int(np.ceil(len(self.sequences) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        return self.sequences[batch_indexes], self.labels[batch_indexes]

# Criando os geradores de dados
train_generator = DataGenerator(X_train, y_train, batch_size)
validation_generator = DataGenerator(X_val, y_val, batch_size)

# Treinamento do modelo
model.fit(train_generator, epochs=epochs, validation_data=validation_generator)

# Função para calcular métricas de avaliação, incluindo a especificidade
def evaluate_model(generator, y_true):
    predictions = model.predict(generator)
    predictions = (predictions > 0.5).astype(int)
    accuracy = accuracy_score(y_true, predictions)
    precision = precision_score(y_true, predictions)
    sensitivity = recall_score(y_true, predictions)
    f1 = f1_score(y_true, predictions)

    # Cálculo da especificidade
    tn, fp, fn, tp = confusion_matrix(y_true, predictions).ravel()
    specificity = tn / (tn + fp)

    return accuracy, precision, sensitivity, f1, specificity

# Avaliação nos arquivos de treino/validação
results = {
    "File": [], "Validation ACC": [], "Validation PRE": [], "Validation SEN": [], "Validation F1S": [], "Validation SPEC": [],
    "Test ACC": [], "Test PRE": [], "Test SEN": [], "Test F1S": [], "Test SPEC": []
}

for test_file in file_paths_train:
    test_data = pd.read_csv(test_file)
    test_sequences = test_data['sequence'].values
    test_labels = test_data['exon_intron_flag'].values

    # Tokenização e padding das sequências de teste
    encoded_test_sequences = tokenizer.texts_to_sequences(test_sequences)
    padded_test_sequences = pad_sequences(encoded_test_sequences, maxlen=max_len, padding='post')

    # Avaliação nos dados de validação
    val_accuracy, val_precision, val_sensitivity, val_f1, val_specificity = evaluate_model(validation_generator, y_val)

    # Avaliação nos dados de teste
    test_predictions = model.predict(padded_test_sequences)
    test_predictions = (test_predictions > 0.5).astype(int)
    test_accuracy = accuracy_score(test_labels, test_predictions)
    test_precision = precision_score(test_labels, test_predictions)
    test_sensitivity = recall_score(test_labels, test_predictions)
    test_f1_score = f1_score(test_labels, test_predictions)

    # Especificidade para o conjunto de teste
    tn, fp, fn, tp = confusion_matrix(test_labels, test_predictions).ravel()
    test_specificity = tn / (tn + fp)

    # Armazenando os resultados
    results["File"].append(test_file)
    results["Validation ACC"].append(val_accuracy * 100)
    results["Validation PRE"].append(val_precision)
    results["Validation SEN"].append(val_sensitivity)
    results["Validation F1S"].append(val_f1)
    results["Validation SPEC"].append(val_specificity)
    results["Test ACC"].append(test_accuracy * 100)
    results["Test PRE"].append(test_precision)
    results["Test SEN"].append(test_sensitivity)
    results["Test F1S"].append(test_f1_score)
    results["Test SPEC"].append(test_specificity)

    print(f'Results for {test_file}:')
    print(f'Validation Accuracy: {val_accuracy * 100:.2f}%')
    print(f'Test Accuracy: {test_accuracy * 100:.2f}%')
    print(f'Test Precision: {test_precision:.2f}')
    print(f'Test Sensitivity: {test_sensitivity:.2f}')
    print(f'Test F1 Score: {test_f1_score:.2f}')
    print(f'Test Specificity: {test_specificity:.2f}')
    print('---')

# Exibindo todos os resultados
results_df = pd.DataFrame(results)
print(results_df)

# Calculando as médias totais
numeric_columns = results_df.select_dtypes(include=[np.number])
total_avg = numeric_columns.mean(axis=0)

# Adicionando as métricas totais ao DataFrame
total_metrics = pd.DataFrame({
    "File": ["Total"],
    "Validation ACC": [total_avg["Validation ACC"]],
    "Validation PRE": [total_avg["Validation PRE"]],
    "Validation SEN": [total_avg["Validation SEN"]],
    "Validation F1S": [total_avg["Validation F1S"]],
    "Validation SPEC": [total_avg["Validation SPEC"]],
    "Test ACC": [total_avg["Test ACC"]],
    "Test PRE": [total_avg["Test PRE"]],
    "Test SEN": [total_avg["Test SEN"]],
    "Test F1S": [total_avg["Test F1S"]],
    "Test SPEC": [total_avg["Test SPEC"]]
})

# Concatenando as métricas totais ao DataFrame de resultados
results_df = pd.concat([results_df, total_metrics], ignore_index=True)
print("Results with Total Metrics:")
print(results_df)


Epoch 1/2


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


499/499 ━━━━━━━━━━━━━━━━━━━━ 366s 719ms/step - accuracy: 0.7976 - loss: 0.4508 - val_accuracy: 0.9327 - val_loss: 0.1912
Epoch 2/2
499/499 ━━━━━━━━━━━━━━━━━━━━ 360s 722ms/step - accuracy: 0.8594 - loss: 0.3252 - val_accuracy: 0.9809 - val_loss: 0.0814
63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 124ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 929ms/step
Results for /content/ANKRD1_test_CORRECTED.csv:
Validation Accuracy: 98.09%
Test Accuracy: 88.24%
Test Precision: 0.89
Test Sensitivity: 0.89
Test F1 Score: 0.89
Test Specificity: 0.88
---
63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 124ms/step
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 109ms/step
Results for /content/B2MFIX_test_CORRECTED.csv:
Validation Accuracy: 98.09%
Test Accuracy: 80.88%
Test Precision: 0.97
Test Sensitivity: 0.70
Test F1 Score: 0.81
Test Specificity: 0.96
---
63/63 ━━━━━━━━━━━━━━━━━━━━ 9s 140ms/step
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 130ms/step
Results for /content/PPIAFIX2_test_CORRECTED.csv:
Validation Accuracy: 98.09%
Test Accuracy: 80.28%
Test Precision: 1.00
Test Sens

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.utils import Sequence
import gc

# Parâmetros
chunk_size = 1000
max_len_limit = 500
batch_size = 16
epochs = 1
random_state = 42

# Caminhos dos arquivos, incluindo o NEB no treinamento
file_paths_train = [
    "/content/ANKRD1_test_CORRECTED.csv",
    "/content/B2MFIX_test_CORRECTED.csv",
    "/content/PPIAFIX2_test_CORRECTED.csv",
    "/content/GADPH_test_CORRECTED.csv",
    "/content/PGK1_test_CORRECTED.csv",
    "/content/RPLA13A_test_CORRECTED.csv",
    "/content/TTN_test_CORRECTED.csv",
    "/content/NEB_test_test_CORRECTED.csv"
]

# Função para processar chunks
def process_chunk(chunk):
    sequences = chunk['sequence'].values
    labels = chunk['exon_intron_flag'].values
    return sequences, labels

# Inicializando o tokenizer
tokenizer = Tokenizer(char_level=True)
sequences_list = []
labels_list = []

# Processando os arquivos CSV em chunks para treino
for file_path in file_paths_train:
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        sequences, labels = process_chunk(chunk)
        tokenizer.fit_on_texts(sequences)
        sequences_list.extend(sequences)
        labels_list.extend(labels)
        gc.collect()

# Tokenização e padding das sequências
encoded_sequences = tokenizer.texts_to_sequences(sequences_list)
max_len = min(max(len(seq) for seq in encoded_sequences), max_len_limit)
padded_sequences = pad_sequences(encoded_sequences, maxlen=max_len, padding='post')
labels = np.array(labels_list, dtype=np.int8)

# Separando os dados em treino (80%), validação (10%) e teste (10%)
X_train, X_temp, y_train, y_temp = train_test_split(padded_sequences, labels, test_size=0.2, random_state=random_state)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=random_state)

from tensorflow.keras.layers import LSTM, Bidirectional

# Construção do modelo RNN com LSTM bidirecional
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 32

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    Bidirectional(LSTM(32, return_sequences=True)),
    Dropout(0.2),
    Bidirectional(LSTM(32)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Classe Data Generator
class DataGenerator(Sequence):
    def __init__(self, sequences, labels, batch_size):
        self.sequences = sequences
        self.labels = labels
        self.batch_size = batch_size
        self.indexes = np.arange(len(self.sequences))

    def __len__(self):
        return int(np.ceil(len(self.sequences) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        return self.sequences[batch_indexes], self.labels[batch_indexes]

# Criando os geradores de dados
train_generator = DataGenerator(X_train, y_train, batch_size)
validation_generator = DataGenerator(X_val, y_val, batch_size)

# Treinamento do modelo
model.fit(train_generator, epochs=epochs, validation_data=validation_generator)

# Função para calcular métricas de avaliação, incluindo a especificidade
def evaluate_model(generator, y_true):
    predictions = model.predict(generator)
    predictions = (predictions > 0.5).astype(int)
    accuracy = accuracy_score(y_true, predictions)
    precision = precision_score(y_true, predictions)
    sensitivity = recall_score(y_true, predictions)
    f1 = f1_score(y_true, predictions)

    # Cálculo da especificidade
    tn, fp, fn, tp = confusion_matrix(y_true, predictions).ravel()
    specificity = tn / (tn + fp)

    return accuracy, precision, sensitivity, f1, specificity

# Avaliação nos dados de validação
val_accuracy, val_precision, val_sensitivity, val_f1, val_specificity = evaluate_model(validation_generator, y_val)

# Avaliação nos dados de teste
test_generator = DataGenerator(X_test, y_test, batch_size)
test_accuracy, test_precision, test_sensitivity, test_f1, test_specificity = evaluate_model(test_generator, y_test)

# Exibindo os resultados de validação e teste
print(f'Validation Accuracy: {val_accuracy * 100:.2f}%')
print(f'Test Accuracy: {test_accuracy * 100:.2f}%')
print(f'Test Precision: {test_precision:.2f}')
print(f'Test Sensitivity (Recall): {test_sensitivity:.2f}')
print(f'Test Specificity: {test_specificity:.2f}')
print(f'Test F1 Score: {test_f1:.2f}')


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


499/499 ━━━━━━━━━━━━━━━━━━━━ 366s 720ms/step - accuracy: 0.7631 - loss: 0.4971 - val_accuracy: 0.8273 - val_loss: 0.3978
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 149ms/step
 1/63 ━━━━━━━━━━━━━━━━━━━━ 7s 123ms/step

/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 122ms/step
Validation Accuracy: 82.73%
Test Accuracy: 80.44%
Test Precision: 0.75
Test Sensitivity (Recall): 0.95
Test Specificity: 0.65
Test F1 Score: 0.83


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.utils import Sequence
import gc

# Parâmetros
chunk_size = 1000
max_len_limit = 500
batch_size = 16
epochs = 20
random_state = 42

# Caminhos dos arquivos, incluindo o NEB no treinamento
file_paths_train = [
    "/content/ANKRD1_test_CORRECTED.csv",
    "/content/B2MFIX_test_CORRECTED.csv",
    "/content/PPIAFIX2_test_CORRECTED.csv",
    "/content/GADPH_test_CORRECTED.csv",
    "/content/PGK1_test_CORRECTED.csv",
    "/content/RPLA13A_test_CORRECTED.csv",
    "/content/TTN_test_CORRECTED.csv",
    "/content/NEB_test_test_CORRECTED.csv"
]

# Função para processar chunks
def process_chunk(chunk):
    sequences = chunk['sequence'].values
    labels = chunk['exon_intron_flag'].values
    return sequences, labels

# Inicializando o tokenizer
tokenizer = Tokenizer(char_level=True)
sequences_list = []
labels_list = []

# Processando os arquivos CSV em chunks para treino
for file_path in file_paths_train:
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        sequences, labels = process_chunk(chunk)
        tokenizer.fit_on_texts(sequences)
        sequences_list.extend(sequences)
        labels_list.extend(labels)
        gc.collect()

# Tokenização e padding das sequências
encoded_sequences = tokenizer.texts_to_sequences(sequences_list)
max_len = min(max(len(seq) for seq in encoded_sequences), max_len_limit)
padded_sequences = pad_sequences(encoded_sequences, maxlen=max_len, padding='post')
labels = np.array(labels_list, dtype=np.int8)

# Separando os dados em treino (80%), validação (10%) e teste (10%)
X_train, X_temp, y_train, y_temp = train_test_split(padded_sequences, labels, test_size=0.2, random_state=random_state)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=random_state)

from tensorflow.keras.layers import LSTM, Bidirectional

# Construção do modelo RNN com LSTM bidirecional
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 32

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    Bidirectional(LSTM(32, return_sequences=True)),
    Dropout(0.2),
    Bidirectional(LSTM(32)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Classe Data Generator
class DataGenerator(Sequence):
    def __init__(self, sequences, labels, batch_size):
        self.sequences = sequences
        self.labels = labels
        self.batch_size = batch_size
        self.indexes = np.arange(len(self.sequences))

    def __len__(self):
        return int(np.ceil(len(self.sequences) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        return self.sequences[batch_indexes], self.labels[batch_indexes]

# Criando os geradores de dados
train_generator = DataGenerator(X_train, y_train, batch_size)
validation_generator = DataGenerator(X_val, y_val, batch_size)

# Treinamento do modelo
model.fit(train_generator, epochs=epochs, validation_data=validation_generator)

# Função para calcular métricas de avaliação, incluindo a especificidade
def evaluate_model(generator, y_true):
    predictions = model.predict(generator)
    predictions = (predictions > 0.5).astype(int)
    accuracy = accuracy_score(y_true, predictions)
    precision = precision_score(y_true, predictions)
    sensitivity = recall_score(y_true, predictions)
    f1 = f1_score(y_true, predictions)

    # Cálculo da especificidade
    tn, fp, fn, tp = confusion_matrix(y_true, predictions).ravel()
    specificity = tn / (tn + fp)

    return accuracy, precision, sensitivity, f1, specificity

# Avaliação nos dados de validação
val_accuracy, val_precision, val_sensitivity, val_f1, val_specificity = evaluate_model(validation_generator, y_val)

# Avaliação nos dados de teste
test_generator = DataGenerator(X_test, y_test, batch_size)
test_accuracy, test_precision, test_sensitivity, test_f1, test_specificity = evaluate_model(test_generator, y_test)

# Exibindo os resultados de validação e teste
print(f'Validation Accuracy: {val_accuracy * 100:.2f}%')
print(f'Test Accuracy: {test_accuracy * 100:.2f}%')
print(f'Test Precision: {test_precision:.2f}')
print(f'Test Sensitivity (Recall): {test_sensitivity:.2f}')
print(f'Test Specificity: {test_specificity:.2f}')
print(f'Test F1 Score: {test_f1:.2f}')


Epoch 1/20


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


499/499 ━━━━━━━━━━━━━━━━━━━━ 368s 722ms/step - accuracy: 0.8017 - loss: 0.4394 - val_accuracy: 0.8062 - val_loss: 0.5566
Epoch 2/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 381s 719ms/step - accuracy: 0.7811 - loss: 0.4719 - val_accuracy: 0.9026 - val_loss: 0.2673
Epoch 3/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 383s 721ms/step - accuracy: 0.9038 - loss: 0.2546 - val_accuracy: 0.9639 - val_loss: 0.1167
Epoch 4/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 359s 719ms/step - accuracy: 0.9282 - loss: 0.1955 - val_accuracy: 0.9367 - val_loss: 0.1807
Epoch 5/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 354s 710ms/step - accuracy: 0.9169 - loss: 0.2246 - val_accuracy: 0.9518 - val_loss: 0.1275
Epoch 6/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 385s 717ms/step - accuracy: 0.9441 - loss: 0.1388 - val_accuracy: 0.9488 - val_loss: 0.1125
Epoch 7/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 382s 715ms/step - accuracy: 0.9685 - loss: 0.0851 - val_accuracy: 0.9859 - val_loss: 0.0602
Epoch 8/20
499/499 ━━━━━━━━━━━━━━━━━━━━ 355s 711ms/step - accuracy: 0.9853 - loss: 0.05

/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


63/63 ━━━━━━━━━━━━━━━━━━━━ 9s 138ms/step
Validation Accuracy: 99.70%
Test Accuracy: 99.60%
Test Precision: 1.00
Test Sensitivity (Recall): 1.00
Test Specificity: 1.00
Test F1 Score: 1.00


In [ ]:
#GRU60

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.utils import Sequence
import gc

# Parâmetros
chunk_size = 1000
max_len_limit = 500
batch_size = 16
epochs = 60
random_state = 42

# Caminhos dos arquivos, incluindo o NEB no treinamento
file_paths_train = [
    "/content/ANKRD1_test_CORRECTED.csv",
    "/content/B2MFIX_test_CORRECTED.csv",
    "/content/PPIAFIX2_test_CORRECTED.csv",
    "/content/GADPH_test_CORRECTED.csv",
    "/content/PGK1_test_CORRECTED.csv",
    "/content/RPLA13A_test_CORRECTED.csv",
    "/content/TTN_test_CORRECTED.csv",
    "/content/NEB_test_test_CORRECTED.csv"
]

# Função para processar chunks
def process_chunk(chunk):
    sequences = chunk['sequence'].values
    labels = chunk['exon_intron_flag'].values
    return sequences, labels

# Inicializando o tokenizer
tokenizer = Tokenizer(char_level=True)
sequences_list = []
labels_list = []

# Processando os arquivos CSV em chunks para treino
for file_path in file_paths_train:
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        sequences, labels = process_chunk(chunk)
        tokenizer.fit_on_texts(sequences)
        sequences_list.extend(sequences)
        labels_list.extend(labels)
        gc.collect()

# Tokenização e padding das sequências
encoded_sequences = tokenizer.texts_to_sequences(sequences_list)
max_len = min(max(len(seq) for seq in encoded_sequences), max_len_limit)
padded_sequences = pad_sequences(encoded_sequences, maxlen=max_len, padding='post')
labels = np.array(labels_list, dtype=np.int8)

# Separando os dados em treino (80%), validação (10%) e teste (10%)
X_train, X_temp, y_train, y_temp = train_test_split(padded_sequences, labels, test_size=0.2, random_state=random_state)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=random_state)

# Construção do modelo RNN com GRU
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 32

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    GRU(32, return_sequences=True),  # Primeira camada GRU
    Dropout(0.2),
    GRU(32),  # Segunda camada GRU
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Classe Data Generator
class DataGenerator(Sequence):
    def __init__(self, sequences, labels, batch_size):
        self.sequences = sequences
        self.labels = labels
        self.batch_size = batch_size
        self.indexes = np.arange(len(self.sequences))

    def __len__(self):
        return int(np.ceil(len(self.sequences) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        return self.sequences[batch_indexes], self.labels[batch_indexes]

# Criando os geradores de dados
train_generator = DataGenerator(X_train, y_train, batch_size)
validation_generator = DataGenerator(X_val, y_val, batch_size)

# Treinamento do modelo
model.fit(train_generator, epochs=epochs, validation_data=validation_generator)

# Função para calcular métricas de avaliação
def evaluate_model(generator, y_true):
    predictions = model.predict(generator)
    predictions = (predictions > 0.5).astype(int)

    # Métricas de classificação
    accuracy = accuracy_score(y_true, predictions)
    precision = precision_score(y_true, predictions)
    sensitivity = recall_score(y_true, predictions)

    # Cálculo da especificidade
    tn = np.sum((y_true == 0) & (predictions == 0))
    fp = np.sum((y_true == 0) & (predictions == 1))
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    # F1-score
    f1 = f1_score(y_true, predictions)

    return accuracy, precision, sensitivity, specificity, f1

# Avaliação no conjunto de teste
test_generator = DataGenerator(X_test, y_test, batch_size)
accuracy, precision, sensitivity, specificity, f1 = evaluate_model(test_generator, y_test)

# Exibindo métricas
print(f'Test Accuracy: {accuracy * 100:.2f}%')
print(f'Test Precision: {precision:.2f}')
print(f'Test Sensitivity (Recall): {sensitivity:.2f}')
print(f'Test Specificity: {specificity:.2f}')
print(f'Test F1 Score: {f1:.2f}')


Epoch 1/60


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


499/499 ━━━━━━━━━━━━━━━━━━━━ 276s 542ms/step - accuracy: 0.6473 - loss: 0.6201 - val_accuracy: 0.4900 - val_loss: 0.6601
Epoch 2/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 312s 523ms/step - accuracy: 0.6343 - loss: 0.6074 - val_accuracy: 0.6687 - val_loss: 0.5891
Epoch 3/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 261s 522ms/step - accuracy: 0.6575 - loss: 0.5811 - val_accuracy: 0.6827 - val_loss: 0.5610
Epoch 4/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 259s 516ms/step - accuracy: 0.6779 - loss: 0.5536 - val_accuracy: 0.6797 - val_loss: 0.5489
Epoch 5/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 257s 516ms/step - accuracy: 0.6697 - loss: 0.5554 - val_accuracy: 0.6837 - val_loss: 0.5475
Epoch 6/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 259s 510ms/step - accuracy: 0.6741 - loss: 0.5473 - val_accuracy: 0.6837 - val_loss: 0.5404
Epoch 7/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 268s 523ms/step - accuracy: 0.6872 - loss: 0.5407 - val_accuracy: 0.9839 - val_loss: 0.0543
Epoch 8/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 258s 514ms/step - accuracy: 0.9829 - loss: 0.06

In [ ]:
#SIMPLERNN60EPOCHSRIGHTFINAL

# Importações
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
from tensorflow.keras.utils import Sequence
import gc
import matplotlib.pyplot as plt

# Parâmetros
chunk_size = 1000
max_len_limit = 500
batch_size = 16+2
Sessão expirada
ARIEL LIMA ABADE BANDEIRA
ariel.bandeira@dcomp.ufs.br
(abre um novo separador)

epochs = 60
random_state = 42

# Caminhos dos arquivos
file_paths_train = [
    "/content/ANKRD1_test_CORRECTED.csv",
    "/content/B2MFIX_test_CORRECTED.csv",
    "/content/PPIAFIX2_test_CORRECTED.csv",
    "/content/GADPH_test_CORRECTED.csv",
    "/content/PGK1_test_CORRECTED.csv",
    "/content/RPLA13A_test_CORRECTED.csv",
    "/content/TTN_test_CORRECTED.csv",
    "/content/NEB_test_test_CORRECTED.csv"
]

# Função para processar chunks
def process_chunk(chunk):
    sequences = chunk['sequence'].values
    labels = chunk['exon_intron_flag'].values
    return sequences, labels

# Inicializando o tokenizer
tokenizer = Tokenizer(char_level=True)
sequences_list = []
labels_list = []

# Processando os arquivos CSV em chunks para treino
for file_path in file_paths_train:
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        sequences, labels = process_chunk(chunk)
        tokenizer.fit_on_texts(sequences)
        sequences_list.extend(sequences)
        labels_list.extend(labels)
        gc.collect()

# Tokenização e padding das sequências
encoded_sequences = tokenizer.texts_to_sequences(sequences_list)
max_len = min(max(len(seq) for seq in encoded_sequences), max_len_limit)
padded_sequences = pad_sequences(encoded_sequences, maxlen=max_len, padding='post')
labels = np.array(labels_list, dtype=np.int8)

# Separando os dados em treino, validação e teste
X_train, X_temp, y_train, y_temp = train_test_split(padded_sequences, labels, test_size=0.2, random_state=random_state)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=random_state)

# Construção do modelo RNN com SimpleRNN
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 32

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    SimpleRNN(32, return_sequences=True),  # Primeira camada SimpleRNN
    Dropout(0.2),
    SimpleRNN(32),  # Segunda camada SimpleRNN
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Classe Data Generator
class DataGenerator(Sequence):
    def __init__(self, sequences, labels, batch_size):
        self.sequences = sequences
        self.labels = labels
        self.batch_size = batch_size
        self.indexes = np.arange(len(self.sequences))

    def __len__(self):
        return int(np.ceil(len(self.sequences) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        return self.sequences[batch_indexes], self.labels[batch_indexes]

# Criando os geradores de dados
train_generator = DataGenerator(X_train, y_train, batch_size)
validation_generator = DataGenerator(X_val, y_val, batch_size)

# Treinamento do modelo
model.fit(train_generator, epochs=epochs, validation_data=validation_generator)

# Função corrigida para calcular métricas de avaliação com precisão de 4 casas decimais e padronização de 0 a 1
def evaluate_model(generator, y_true):
    # Gera previsões
    predictions = model.predict(generator)
    predictions = (predictions > 0.5).astype(int).flatten()  # Transforma em array 1D

    # Exibe o tamanho dos dados em predictions e y_true
    print(f'Quantidade de previsões (predictions): {len(predictions)}')
    print(f'Quantidade de valores verdadeiros (y_true): {len(y_true)}')

    # Calcula TP, TN, FP, FN
    tp = np.sum((y_true == 1) & (predictions == 1))
    tn = np.sum((y_true == 0) & (predictions == 0))
    fp = np.sum((y_true == 0) & (predictions == 1))
    fn = np.sum((y_true == 1) & (predictions == 0))

    # Métricas de classificação
    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    # Retorna as métricas com precisão de 4 casas decimais
    return (round(accuracy, 4), round(precision, 4), round(recall, 4),
            round(specificity, 4), round(f1, 4), tp, tn, fp, fn)

# Avaliação no conjunto de teste
test_generator = DataGenerator(X_test, y_test, batch_size)
accuracy, precision, sensitivity, specificity, f1, tp, tn, fp, fn = evaluate_model(test_generator, y_test)

# Exibindo as métricas
print(f'Test Accuracy: {accuracy:.4f}')
print(f'Test Precision: {precision:.4f}')
print(f'Test Sensitivity (Recall): {sensitivity:.4f}')
print(f'Test Specificity: {specificity:.4f}')
print(f'Test F1-Score: {f1:.4f}')
print(f'True Positives: {tp}')
print(f'True Negatives: {tn}')
print(f'False Positives: {fp}')
print(f'False Negatives: {fn}')


Epoch 1/60


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


499/499 ━━━━━━━━━━━━━━━━━━━━ 107s 206ms/step - accuracy: 0.6253 - loss: 0.6462 - val_accuracy: 0.6135 - val_loss: 0.6004
Epoch 2/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 142s 206ms/step - accuracy: 0.6398 - loss: 0.6012 - val_accuracy: 0.5000 - val_loss: 0.6207
Epoch 3/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 101s 202ms/step - accuracy: 0.6685 - loss: 0.5709 - val_accuracy: 0.5753 - val_loss: 0.6147
Epoch 4/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 103s 207ms/step - accuracy: 0.6576 - loss: 0.5833 - val_accuracy: 0.6566 - val_loss: 0.5005
Epoch 5/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 107s 213ms/step - accuracy: 0.6612 - loss: 0.5155 - val_accuracy: 0.6797 - val_loss: 0.4735
Epoch 6/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 138s 207ms/step - accuracy: 0.6656 - loss: 0.4980 - val_accuracy: 0.6878 - val_loss: 0.4873
Epoch 7/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 140s 204ms/step - accuracy: 0.6498 - loss: 0.5684 - val_accuracy: 0.6687 - val_loss: 0.5140
Epoch 8/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 141s 201ms/step - accuracy: 0.6588 - loss: 0.51

In [ ]:
#GRU60EPOCHSRIGHTFINAL

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.utils import Sequence
import gc

# Parâmetros
chunk_size = 1000
max_len_limit = 500
batch_size = 16
epochs = 60
random_state = 42

# Caminhos dos arquivos, incluindo o NEB no treinamento
file_paths_train = [
    "/content/ANKRD1_test_CORRECTED.csv",
    "/content/B2MFIX_test_CORRECTED.csv",
    "/content/PPIAFIX2_test_CORRECTED.csv",
    "/content/GADPH_test_CORRECTED.csv",
    "/content/PGK1_test_CORRECTED.csv",
    "/content/RPLA13A_test_CORRECTED.csv",
    "/content/TTN_test_CORRECTED.csv",
    "/content/NEB_test_test_CORRECTED.csv"
]

# Função para processar chunks
def process_chunk(chunk):
    sequences = chunk['sequence'].values
    labels = chunk['exon_intron_flag'].values
    return sequences, labels

# Inicializando o tokenizer
tokenizer = Tokenizer(char_level=True)
sequences_list = []
labels_list = []

# Processando os arquivos CSV em chunks para treino
for file_path in file_paths_train:
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        sequences, labels = process_chunk(chunk)
        tokenizer.fit_on_texts(sequences)
        sequences_list.extend(sequences)
        labels_list.extend(labels)
        gc.collect()

# Tokenização e padding das sequências
encoded_sequences = tokenizer.texts_to_sequences(sequences_list)
max_len = min(max(len(seq) for seq in encoded_sequences), max_len_limit)
padded_sequences = pad_sequences(encoded_sequences, maxlen=max_len, padding='post')
labels = np.array(labels_list, dtype=np.int8)

# Separando os dados em treino (80%), validação (10%) e teste (10%)
X_train, X_temp, y_train, y_temp = train_test_split(padded_sequences, labels, test_size=0.2, random_state=random_state)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=random_state)

# Construção do modelo RNN com GRU
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 32

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    GRU(32, return_sequences=True),  # Primeira camada GRU
    Dropout(0.2),
    GRU(32),  # Segunda camada GRU
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Classe Data Generator
class DataGenerator(Sequence):
    def __init__(self, sequences, labels, batch_size):
        self.sequences = sequences
        self.labels = labels
        self.batch_size = batch_size
        self.indexes = np.arange(len(self.sequences))

    def __len__(self):
        return int(np.ceil(len(self.sequences) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        return self.sequences[batch_indexes], self.labels[batch_indexes]

# Criando os geradores de dados
train_generator = DataGenerator(X_train, y_train, batch_size)
validation_generator = DataGenerator(X_val, y_val, batch_size)

# Treinamento do modelo
model.fit(train_generator, epochs=epochs, validation_data=validation_generator)

# Função corrigida para calcular métricas de avaliação com precisão de 4 casas decimais e padronização de 0 a 1
def evaluate_model(generator, y_true):
    # Gera previsões
    predictions = model.predict(generator)
    predictions = (predictions > 0.5).astype(int).flatten()  # Transforma em array 1D

    # Exibe o tamanho dos dados em predictions e y_true
    print(f'Quantidade de previsões (predictions): {len(predictions)}')
    print(f'Quantidade de valores verdadeiros (y_true): {len(y_true)}')

    # Calcula TP, TN, FP, FN
    tp = np.sum((y_true == 1) & (predictions == 1))
    tn = np.sum((y_true == 0) & (predictions == 0))
    fp = np.sum((y_true == 0) & (predictions == 1))
    fn = np.sum((y_true == 1) & (predictions == 0))

    # Métricas de classificação
    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    # Retorna as métricas com precisão de 4 casas decimais
    return (round(accuracy, 4), round(precision, 4), round(recall, 4),
            round(specificity, 4), round(f1, 4), tp, tn, fp, fn)

# Avaliação no conjunto de teste
test_generator = DataGenerator(X_test, y_test, batch_size)
accuracy, precision, sensitivity, specificity, f1, tp, tn, fp, fn = evaluate_model(test_generator, y_test)

# Exibindo as métricas
print(f'Test Accuracy: {accuracy:.4f}')
print(f'Test Precision: {precision:.4f}')
print(f'Test Sensitivity (Recall): {sensitivity:.4f}')
print(f'Test Specificity: {specificity:.4f}')
print(f'Test F1-Score: {f1:.4f}')
print(f'True Positives: {tp}')
print(f'True Negatives: {tn}')
print(f'False Positives: {fp}')
print(f'False Negatives: {fn}')





Epoch 1/60


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


499/499 ━━━━━━━━━━━━━━━━━━━━ 224s 441ms/step - accuracy: 0.6733 - loss: 0.5982 - val_accuracy: 0.6757 - val_loss: 0.5722
Epoch 2/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 258s 432ms/step - accuracy: 0.6519 - loss: 0.5903 - val_accuracy: 0.6797 - val_loss: 0.5655
Epoch 3/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 265s 438ms/step - accuracy: 0.6842 - loss: 0.5640 - val_accuracy: 0.6807 - val_loss: 0.5561
Epoch 4/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 217s 434ms/step - accuracy: 0.6752 - loss: 0.5663 - val_accuracy: 0.6807 - val_loss: 0.5544
Epoch 5/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 215s 430ms/step - accuracy: 0.6744 - loss: 0.5573 - val_accuracy: 0.6807 - val_loss: 0.5518
Epoch 6/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 263s 434ms/step - accuracy: 0.6693 - loss: 0.5727 - val_accuracy: 0.9799 - val_loss: 0.1042
Epoch 7/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 218s 436ms/step - accuracy: 0.9636 - loss: 0.1343 - val_accuracy: 0.9839 - val_loss: 0.0549
Epoch 8/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 262s 436ms/step - accuracy: 0.9833 - loss: 0.06

In [ ]:
#GRU60EPOCHSRIGHTFINAL

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.utils import Sequence
import gc

# Parâmetros
chunk_size = 1000
max_len_limit = 500
batch_size = 16
epochs = 1
random_state = 42

# Caminhos dos arquivos, incluindo o NEB no treinamento
file_paths_train = [
    "/content/ANKRD1_test_CORRECTED.csv",
    "/content/B2MFIX_test_CORRECTED.csv",
    "/content/PPIAFIX2_test_CORRECTED.csv",
    "/content/GADPH_test_CORRECTED.csv",
    "/content/PGK1_test_CORRECTED.csv",
    "/content/RPLA13A_test_CORRECTED.csv",
    "/content/TTN_test_CORRECTED.csv",
    "/content/NEB_test_test_CORRECTED.csv"
]

# Função para processar chunks
def process_chunk(chunk):
    sequences = chunk['sequence'].values
    labels = chunk['exon_intron_flag'].values
    return sequences, labels

# Inicializando o tokenizer
tokenizer = Tokenizer(char_level=True)
sequences_list = []
labels_list = []

# Processando os arquivos CSV em chunks para treino
for file_path in file_paths_train:
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        sequences, labels = process_chunk(chunk)
        tokenizer.fit_on_texts(sequences)
        sequences_list.extend(sequences)
        labels_list.extend(labels)
        gc.collect()

# Tokenização e padding das sequências
encoded_sequences = tokenizer.texts_to_sequences(sequences_list)
max_len = min(max(len(seq) for seq in encoded_sequences), max_len_limit)
padded_sequences = pad_sequences(encoded_sequences, maxlen=max_len, padding='post')
labels = np.array(labels_list, dtype=np.int8)

# Separando os dados em treino (80%), validação (10%) e teste (10%)
X_train, X_temp, y_train, y_temp = train_test_split(padded_sequences, labels, test_size=0.2, random_state=random_state)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=random_state)

# Construção do modelo RNN com GRU
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 32

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    GRU(32, return_sequences=True),  # Primeira camada GRU
    Dropout(0.2),
    GRU(32),  # Segunda camada GRU
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Classe Data Generator
class DataGenerator(Sequence):
    def __init__(self, sequences, labels, batch_size):
        self.sequences = sequences
        self.labels = labels
        self.batch_size = batch_size
        self.indexes = np.arange(len(self.sequences))

    def __len__(self):
        return int(np.ceil(len(self.sequences) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        return self.sequences[batch_indexes], self.labels[batch_indexes]

# Criando os geradores de dados
train_generator = DataGenerator(X_train, y_train, batch_size)
validation_generator = DataGenerator(X_val, y_val, batch_size)

# Treinamento do modelo
model.fit(train_generator, epochs=epochs, validation_data=validation_generator)

# Função corrigida para calcular métricas de avaliação com precisão de 4 casas decimais e padronização de 0 a 1
def evaluate_model(generator, y_true):
    # Gera previsões
    predictions = model.predict(generator)
    predictions = (predictions > 0.5).astype(int).flatten()  # Transforma em array 1D

    # Exibe o tamanho dos dados em predictions e y_true
    print(f'Quantidade de previsões (predictions): {len(predictions)}')
    print(f'Quantidade de valores verdadeiros (y_true): {len(y_true)}')

    # Calcula TP, TN, FP, FN
    tp = np.sum((y_true == 1) & (predictions == 1))
    tn = np.sum((y_true == 0) & (predictions == 0))
    fp = np.sum((y_true == 0) & (predictions == 1))
    fn = np.sum((y_true == 1) & (predictions == 0))

    # Métricas de classificação
    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    specificity = tn / (tn + fp)
    f1 = 2 * (precision * recall) / (precision + recall)

    # Retorna as métricas com precisão de 4 casas decimais
    return (round(accuracy, 4), round(precision, 4), round(recall, 4),
            round(specificity, 4), round(f1, 4), tp, tn, fp, fn)

# Avaliação no conjunto de teste
test_generator = DataGenerator(X_test, y_test, batch_size)
accuracy, precision, sensitivity, specificity, f1, tp, tn, fp, fn = evaluate_model(test_generator, y_test)

# Exibindo as métricas
print(f'Test Accuracy: {accuracy:.4f}')
print(f'Test Precision: {precision:.4f}')
print(f'Test Sensitivity (Recall): {sensitivity:.4f}')
print(f'Test Specificity: {specificity:.4f}')
print(f'Test F1-Score: {f1:.4f}')
print(f'True Positives: {tp}')
print(f'True Negatives: {tn}')
print(f'False Positives: {fp}')
print(f'False Negatives: {fn}')





/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


499/499 ━━━━━━━━━━━━━━━━━━━━ 226s 441ms/step - accuracy: 0.6420 - loss: 0.6266 - val_accuracy: 0.6797 - val_loss: 0.5999
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 73ms/step
Quantidade de previsões (predictions): 997
Quantidade de valores verdadeiros (y_true): 997
Test Accuracy: 0.6740
Test Precision: 0.6185
Test Sensitivity (Recall): 0.9788
Test Specificity: 0.3417
Test F1-Score: 0.7580
True Positives: 509
True Negatives: 163
False Positives: 314
False Negatives: 11


In [ ]:
#GRU30EPOCHSRIGHTFINAL

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.utils import Sequence
import gc

# Parâmetros
chunk_size = 1000
max_len_limit = 500
batch_size = 16
epochs = 30
random_state = 42

# Caminhos dos arquivos, incluindo o NEB no treinamento
file_paths_train = [
    "/content/ANKRD1_test_CORRECTED.csv",
    "/content/B2MFIX_test_CORRECTED.csv",
    "/content/PPIAFIX2_test_CORRECTED.csv",
    "/content/GADPH_test_CORRECTED.csv",
    "/content/PGK1_test_CORRECTED.csv",
    "/content/RPLA13A_test_CORRECTED.csv",
    "/content/TTN_test_CORRECTED.csv",
    "/content/NEB_test_test_CORRECTED.csv"
]

# Função para processar chunks
def process_chunk(chunk):
    sequences = chunk['sequence'].values
    labels = chunk['exon_intron_flag'].values
    return sequences, labels

# Inicializando o tokenizer
tokenizer = Tokenizer(char_level=True)
sequences_list = []
labels_list = []

# Processando os arquivos CSV em chunks para treino
for file_path in file_paths_train:
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        sequences, labels = process_chunk(chunk)
        tokenizer.fit_on_texts(sequences)
        sequences_list.extend(sequences)
        labels_list.extend(labels)
        gc.collect()

# Tokenização e padding das sequências
encoded_sequences = tokenizer.texts_to_sequences(sequences_list)
max_len = min(max(len(seq) for seq in encoded_sequences), max_len_limit)
padded_sequences = pad_sequences(encoded_sequences, maxlen=max_len, padding='post')
labels = np.array(labels_list, dtype=np.int8)

# Separando os dados em treino (80%), validação (10%) e teste (10%)
X_train, X_temp, y_train, y_temp = train_test_split(padded_sequences, labels, test_size=0.2, random_state=random_state)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=random_state)

# Construção do modelo RNN com GRU
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 32

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    GRU(32, return_sequences=True),  # Primeira camada GRU
    Dropout(0.2),
    GRU(32),  # Segunda camada GRU
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Classe Data Generator
class DataGenerator(Sequence):
    def __init__(self, sequences, labels, batch_size):
        self.sequences = sequences
        self.labels = labels
        self.batch_size = batch_size
        self.indexes = np.arange(len(self.sequences))

    def __len__(self):
        return int(np.ceil(len(self.sequences) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        return self.sequences[batch_indexes], self.labels[batch_indexes]

# Criando os geradores de dados
train_generator = DataGenerator(X_train, y_train, batch_size)
validation_generator = DataGenerator(X_val, y_val, batch_size)

# Treinamento do modelo
model.fit(train_generator, epochs=epochs, validation_data=validation_generator)

# Função corrigida para calcular métricas de avaliação com precisão de 4 casas decimais e padronização de 0 a 1
def evaluate_model(generator, y_true):
    # Gera previsões
    predictions = model.predict(generator)
    predictions = (predictions > 0.5).astype(int).flatten()  # Transforma em array 1D

    # Exibe o tamanho dos dados em predictions e y_true
    print(f'Quantidade de previsões (predictions): {len(predictions)}')
    print(f'Quantidade de valores verdadeiros (y_true): {len(y_true)}')

    # Calcula TP, TN, FP, FN
    tp = np.sum((y_true == 1) & (predictions == 1))
    tn = np.sum((y_true == 0) & (predictions == 0))
    fp = np.sum((y_true == 0) & (predictions == 1))
    fn = np.sum((y_true == 1) & (predictions == 0))

    # Métricas de classificação
    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    # Retorna as métricas com precisão de 4 casas decimais
    return (round(accuracy, 4), round(precision, 4), round(recall, 4),
            round(specificity, 4), round(f1, 4), tp, tn, fp, fn)

# Avaliação no conjunto de teste
test_generator = DataGenerator(X_test, y_test, batch_size)
accuracy, precision, sensitivity, specificity, f1, tp, tn, fp, fn = evaluate_model(test_generator, y_test)

# Exibindo as métricas
print(f'Test Accuracy: {accuracy:.4f}')
print(f'Test Precision: {precision:.4f}')
print(f'Test Sensitivity (Recall): {sensitivity:.4f}')
print(f'Test Specificity: {specificity:.4f}')
print(f'Test F1-Score: {f1:.4f}')
print(f'True Positives: {tp}')
print(f'True Negatives: {tn}')
print(f'False Positives: {fp}')
print(f'False Negatives: {fn}')





Epoch 1/30


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


499/499 ━━━━━━━━━━━━━━━━━━━━ 235s 461ms/step - accuracy: 0.6418 - loss: 0.6223 - val_accuracy: 0.6456 - val_loss: 0.6261
Epoch 2/30
499/499 ━━━━━━━━━━━━━━━━━━━━ 253s 443ms/step - accuracy: 0.6529 - loss: 0.5910 - val_accuracy: 0.6677 - val_loss: 0.5699
Epoch 3/30
499/499 ━━━━━━━━━━━━━━━━━━━━ 261s 442ms/step - accuracy: 0.6799 - loss: 0.5611 - val_accuracy: 0.6687 - val_loss: 0.5606
Epoch 4/30
499/499 ━━━━━━━━━━━━━━━━━━━━ 262s 441ms/step - accuracy: 0.6696 - loss: 0.5636 - val_accuracy: 0.6677 - val_loss: 0.5599
Epoch 5/30
499/499 ━━━━━━━━━━━━━━━━━━━━ 262s 442ms/step - accuracy: 0.6806 - loss: 0.5529 - val_accuracy: 0.6657 - val_loss: 0.5604
Epoch 6/30
499/499 ━━━━━━━━━━━━━━━━━━━━ 222s 445ms/step - accuracy: 0.6704 - loss: 0.5556 - val_accuracy: 0.6657 - val_loss: 0.5737
Epoch 7/30
499/499 ━━━━━━━━━━━━━━━━━━━━ 220s 442ms/step - accuracy: 0.6813 - loss: 0.5426 - val_accuracy: 0.6687 - val_loss: 0.5551
Epoch 8/30
499/499 ━━━━━━━━━━━━━━━━━━━━ 220s 441ms/step - accuracy: 0.6737 - loss: 0.54

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout, Bidirectional, LSTM
from tensorflow.keras.utils import Sequence
import gc

# Parâmetros
chunk_size = 1000
max_len_limit = 500
batch_size = 16
epochs = 60
random_state = 42

# Caminhos dos arquivos, incluindo o NEB no treinamento
file_paths_train = [
    "/content/ANKRD1_test_CORRECTED.csv",
    "/content/B2MFIX_test_CORRECTED.csv",
    "/content/PPIAFIX2_test_CORRECTED.csv",
    "/content/GADPH_test_CORRECTED.csv",
    "/content/PGK1_test_CORRECTED.csv",
    "/content/RPLA13A_test_CORRECTED.csv",
    "/content/TTN_test_CORRECTED.csv",
    "/content/NEB_test_test_CORRECTED.csv"
]

# Função para processar chunks
def process_chunk(chunk):
    sequences = chunk['sequence'].values
    labels = chunk['exon_intron_flag'].values
    return sequences, labels

# Inicializando o tokenizer
tokenizer = Tokenizer(char_level=True)
sequences_list = []
labels_list = []

# Processando os arquivos CSV em chunks para treino
for file_path in file_paths_train:
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        sequences, labels = process_chunk(chunk)
        tokenizer.fit_on_texts(sequences)
        sequences_list.extend(sequences)
        labels_list.extend(labels)
        gc.collect()

# Tokenização e padding das sequências
encoded_sequences = tokenizer.texts_to_sequences(sequences_list)
max_len = min(max(len(seq) for seq in encoded_sequences), max_len_limit)
padded_sequences = pad_sequences(encoded_sequences, maxlen=max_len, padding='post')
labels = np.array(labels_list, dtype=np.int8)

# Separando os dados em treino (80%), validação (10%) e teste (10%)
X_train, X_temp, y_train, y_temp = train_test_split(padded_sequences, labels, test_size=0.2, random_state=random_state)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=random_state)

# Construção do modelo RNN com Bi-LSTM
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 64

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    Bidirectional(LSTM(64, return_sequences=True)),  # Primeira camada Bi-LSTM com mais unidades
    Dropout(0.2),
    Bidirectional(LSTM(64, return_sequences=True)),  # Segunda camada Bi-LSTM para capturar mais padrões
    Dropout(0.2),
    Bidirectional(LSTM(32)),  # Terceira camada Bi-LSTM para refinar padrões aprendidos
    Dropout(0.2),
    Dense(128, activation='relu'),  # Camada Dense maior para maior capacidade de aprendizado
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Classe Data Generator
class DataGenerator(Sequence):
    def __init__(self, sequences, labels, batch_size):
        self.sequences = sequences
        self.labels = labels
        self.batch_size = batch_size
        self.indexes = np.arange(len(self.sequences))

    def __len__(self):
        return int(np.ceil(len(self.sequences) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        return self.sequences[batch_indexes], self.labels[batch_indexes]

# Criando os geradores de dados
train_generator = DataGenerator(X_train, y_train, batch_size)
validation_generator = DataGenerator(X_val, y_val, batch_size)

# Treinamento do modelo
model.fit(train_generator, epochs=epochs, validation_data=validation_generator)

# Função para calcular métricas de avaliação com precisão de 4 casas decimais
def evaluate_model(generator, y_true):
    predictions = model.predict(generator)
    predictions = (predictions > 0.5).astype(int).flatten()

    tp = np.sum((y_true == 1) & (predictions == 1))
    tn = np.sum((y_true == 0) & (predictions == 0))
    fp = np.sum((y_true == 0) & (predictions == 1))
    fn = np.sum((y_true == 1) & (predictions == 0))

    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return (round(accuracy, 4), round(precision, 4), round(recall, 4),
            round(specificity, 4), round(f1, 4), tp, tn, fp, fn)

# Avaliação no conjunto de teste
test_generator = DataGenerator(X_test, y_test, batch_size)
accuracy, precision, sensitivity, specificity, f1, tp, tn, fp, fn = evaluate_model(test_generator, y_test)

# Exibindo as métricas
print(f'Test Accuracy: {accuracy:.4f}')
print(f'Test Precision: {precision:.4f}')
print(f'Test Sensitivity (Recall): {sensitivity:.4f}')
print(f'Test Specificity: {specificity:.4f}')
print(f'Test F1-Score: {f1:.4f}')
print(f'True Positives: {tp}')
print(f'True Negatives: {tn}')
print(f'False Positives: {fp}')
print(f'False Negatives: {fn}')


Epoch 1/60


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


499/499 ━━━━━━━━━━━━━━━━━━━━ 637s 1s/step - accuracy: 0.7629 - loss: 0.4856 - val_accuracy: 0.8745 - val_loss: 0.4846
Epoch 2/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 681s 1s/step - accuracy: 0.8739 - loss: 0.3010 - val_accuracy: 0.9699 - val_loss: 0.0838
Epoch 3/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 694s 1s/step - accuracy: 0.9664 - loss: 0.1085 - val_accuracy: 0.9789 - val_loss: 0.0670
Epoch 4/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 675s 1s/step - accuracy: 0.9762 - loss: 0.0852 - val_accuracy: 0.9880 - val_loss: 0.0500
Epoch 5/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 679s 1s/step - accuracy: 0.9837 - loss: 0.0578 - val_accuracy: 0.9819 - val_loss: 0.0566
Epoch 6/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 677s 1s/step - accuracy: 0.9827 - loss: 0.0597 - val_accuracy: 0.9910 - val_loss: 0.0433
Epoch 7/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 622s 1s/step - accuracy: 0.9937 - loss: 0.0278 - val_accuracy: 0.9940 - val_loss: 0.0162
Epoch 8/60
499/499 ━━━━━━━━━━━━━━━━━━━━ 621s 1s/step - accuracy: 0.9919 - loss: 0.0274 - val_accuracy: 0.994

In [ ]:
#SIMPLERNN30EPOCHSBASELINEEVALUATION

# Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
from tensorflow.keras.utils import Sequence
import gc
import matplotlib.pyplot as plt

# Parameters
chunk_size = 1000
max_len_limit = 500
batch_size = 16
epochs = 30
random_state = 42

# File paths
file_paths_train = [
    "/content/ANKRD1_test_CORRECTED.csv",
    "/content/B2MFIX_test_CORRECTED.csv",
    "/content/PPIAFIX2_test_CORRECTED.csv",
    "/content/GADPH_test_CORRECTED.csv",
    "/content/PGK1_test_CORRECTED.csv",
    "/content/RPLA13A_test_CORRECTED.csv",
    "/content/TTN_test_CORRECTED.csv",
    "/content/NEB_test_test_CORRECTED.csv"
]

# Function to process chunks
def process_chunk(chunk):
    sequences = chunk['sequence'].values
    labels = chunk['exon_intron_flag'].values
    return sequences, labels

# Initializing the tokenizer
tokenizer = Tokenizer(char_level=True)
sequences_list = []
labels_list = []

# Processing CSV files in chunks for training
for file_path in file_paths_train:
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        sequences, labels = process_chunk(chunk)
        tokenizer.fit_on_texts(sequences)
        sequences_list.extend(sequences)
        labels_list.extend(labels)
        gc.collect()

# Tokenizing and padding sequences
encoded_sequences = tokenizer.texts_to_sequences(sequences_list)
max_len = min(max(len(seq) for seq in encoded_sequences), max_len_limit)
padded_sequences = pad_sequences(encoded_sequences, maxlen=max_len, padding='post')
labels = np.array(labels_list, dtype=np.int8)

# Splitting data into training, validation, and testing sets
X_train, X_temp, y_train, y_temp = train_test_split(padded_sequences, labels, test_size=0.2, random_state=random_state)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=random_state)

# RNN model with SimpleRNN
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 32

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    SimpleRNN(32, return_sequences=True),  # First SimpleRNN layer
    Dropout(0.2),
    SimpleRNN(32),  # Second SimpleRNN layer
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Data Generator class
class DataGenerator(Sequence):
    def __init__(self, sequences, labels, batch_size):
        self.sequences = sequences
        self.labels = labels
        self.batch_size = batch_size
        self.indexes = np.arange(len(self.sequences))

    def __len__(self):
        return int(np.ceil(len(self.sequences) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        return self.sequences[batch_indexes], self.labels[batch_indexes]

# Creating data generators
train_generator = DataGenerator(X_train, y_train, batch_size)
validation_generator = DataGenerator(X_val, y_val, batch_size)

# Model training
model.fit(train_generator, epochs=epochs, validation_data=validation_generator)

# Function to calculate evaluation metrics
def evaluate_model(generator, y_true):
    # Generate predictions
    predictions = model.predict(generator)
    predictions = (predictions > 0.5).astype(int).flatten()

    # Display data sizes in predictions and y_true
    print(f'Number of predictions: {len(predictions)}')
    print(f'Number of true values: {len(y_true)}')

    # Calculate TP, TN, FP, FN
    tp = np.sum((y_true == 1) & (predictions == 1))
    tn = np.sum((y_true == 0) & (predictions == 0))
    fp = np.sum((y_true == 0) & (predictions == 1))
    fn = np.sum((y_true == 1) & (predictions == 0))

    # Classification metrics
    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    specificity = tn / (tn + fp)
    f1 = 2 * (precision * recall) / (precision + recall)
    # Return metrics rounded to 4 decimal places
    return (round(accuracy, 4), round(precision, 4), round(recall, 4),
            round(specificity, 4), round(f1, 4), tp, tn, fp, fn)

# Evaluation on the test set
test_generator = DataGenerator(X_test, y_test, batch_size)
accuracy, precision, sensitivity, specificity, f1, tp, tn, fp, fn = evaluate_model(test_generator, y_test)

# Display metrics
print(f'Test Accuracy: {accuracy:.4f}')
print(f'Test Precision: {precision:.4f}')
print(f'Test Sensitivity (Recall): {sensitivity:.4f}')
print(f'Test Specificity: {specificity:.4f}')
print(f'Test F1-Score: {f1:.4f}')
print(f'True Positives: {tp}')
print(f'True Negatives: {tn}')
print(f'False Positives: {fp}')
print(f'False Negatives: {fn}')

Epoch 1/30


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


499/499 ━━━━━━━━━━━━━━━━━━━━ 126s 244ms/step - accuracy: 0.5849 - loss: 0.6613 - val_accuracy: 0.6817 - val_loss: 0.5704
Epoch 2/30
499/499 ━━━━━━━━━━━━━━━━━━━━ 118s 236ms/step - accuracy: 0.6993 - loss: 0.5552 - val_accuracy: 0.6767 - val_loss: 0.5796
Epoch 3/30
499/499 ━━━━━━━━━━━━━━━━━━━━ 122s 245ms/step - accuracy: 0.6657 - loss: 0.5640 - val_accuracy: 0.6827 - val_loss: 0.5563
Epoch 4/30
499/499 ━━━━━━━━━━━━━━━━━━━━ 141s 244ms/step - accuracy: 0.6835 - loss: 0.5488 - val_accuracy: 0.6837 - val_loss: 0.5513
Epoch 5/30
499/499 ━━━━━━━━━━━━━━━━━━━━ 119s 238ms/step - accuracy: 0.6826 - loss: 0.5411 - val_accuracy: 0.6817 - val_loss: 0.5525
Epoch 6/30
499/499 ━━━━━━━━━━━━━━━━━━━━ 125s 251ms/step - accuracy: 0.6888 - loss: 0.5357 - val_accuracy: 0.6817 - val_loss: 0.5512
Epoch 7/30
499/499 ━━━━━━━━━━━━━━━━━━━━ 135s 236ms/step - accuracy: 0.6839 - loss: 0.5407 - val_accuracy: 0.6807 - val_loss: 0.5495
Epoch 8/30
499/499 ━━━━━━━━━━━━━━━━━━━━ 144s 241ms/step - accuracy: 0.6805 - loss: 0.54